In [18]:
from mlflow.tracking import MlflowClient
import mlflow

from sklearn.metrics import root_mean_squared_error
import pandas as pd
import pickle

In [2]:
DB_PATH = "/workspaces/mlops-zoomcamp/02-experiment-tracking/mlflow.db"

MLFLOW_TRACKING_URI = f"sqlite:///{DB_PATH}"

In [3]:
# mlflow ui --backend-store-uri sqlite:////workspaces/mlops-zoomcamp/02-experiment-tracking/mlflow.db

In [4]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [5]:
client.search_experiments()

[<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/2', creation_time=1776672140948, experiment_id='2', last_update_time=1776672140948, lifecycle_stage='active', name='my-cool-experiment', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1776061263776, experiment_id='1', last_update_time=1776061263776, lifecycle_stage='active', name='duration-prediction', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/0', creation_time=1776061263770, experiment_id='0', last_update_time=1776061263770, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

In [6]:
# client.create_experiment(name="my-cool-experiment")

In [7]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids="1",
    filter_string="metrics.rmse <6.8 ",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=["metrics.rmse ASC"]

)

for run in runs:
    print(f"run_id: {run.info.run_id}, rmse: {run.data.metrics['rmse']: 4f}")

run_id: d6a6e3a4688c4080ae99754d47d525cc, rmse:  6.318446
run_id: 15ce7961a07148a9bf74db44a6405060, rmse:  6.391028
run_id: 9924c0db8b724238b1f7fd76963ba905, rmse:  6.417410
run_id: 650cf1166a1c45dcb7bdfade8143fc41, rmse:  6.478640
run_id: 59c9635e7659471cade01e548ad39dfe, rmse:  6.479841
run_id: bd3400c9c7174de08d8d85e3bc4fc0ef, rmse:  6.593269
run_id: 61efa9ca47e84c87acf975b0a9b1e286, rmse:  6.600157


### Interacting with the Model Registry

In this section We will use the `MlflowClient` instance to:

1. Register a new version for the experiment `nyc-taxi-regressor`
2. Retrieve the latests versions of the model `nyc-taxi-regressor` and check that a new version `4` was created.
3. Transition the version `4` to "Staging" and adding annotations to it.

In [26]:
model_name = "nyc-taxi-regressor"

In [ ]:
run_id = "c34e8acacfe647a2a82320a6687d6872"
model_uri_to_register = f"runs:/{run_id}/models_mlflow"
mlflow.register_model(model_uri=model_uri_to_register, name=model_name)


Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/04/20 08:12:53 WARNING mlflow.tracking._model_registry.fluent: Run with id c34e8acacfe647a2a82320a6687d6872 has no artifacts at artifact path 'models_mlflow', registering model based on models:/m-f5ee30b061284ed098a56d94be7870d9 instead


Created version '4' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1776672773453, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:/m-f5ee30b061284ed098a56d94be7870d9', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>

In [8]:
# Trasition model from one stage to another
client.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1776669987288, deployment_job_id=None, deployment_job_state=None, description='NYC Taxi duration prediction', last_updated_timestamp=1776673552234, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1776669987379, current_stage='Production', deployment_job_state=None, description='', last_updated_timestamp=1776673383907, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='59c9635e7659471cade01e548ad39dfe', run_link='', source='models:/m-a8a4ca6313bf49f992f2447a0257907f', status='READY', status_message=None, tags={'estimator name': 'xgboost'}, user_id=None, version=1, workspace='default'>,
  <ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='Staging', deployment_job_state=None, description='This model version (4) was transitioned to Staging on 2026-04-20 08:26:00', last_updated_timestamp=1776673560653, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id

In [ ]:
latests_versions = client.get_latest_versions(name=model_name) # returns latest version for each stage

for version in latests_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Production
version: 4, stage: Staging


/tmp/ipykernel_4718/2245416359.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latests_versions = client.get_latest_versions(name="nyc-taxi-regressor") # returns latest version for each stage


In [ ]:
version = 2
stage = "Staging"
client.transition_model_version_stage(name=model_name,
                                      version=version,
                                      stage=stage,
                                      archive_existing_versions=False)


/tmp/ipykernel_4718/1080756545.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(name="nyc-taxi-regressor",


<ModelVersion: aliases=[], creation_timestamp=1776670068425, current_stage='Staging', deployment_job_state=None, description='', last_updated_timestamp=1776750634230, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='d5efa482a4c94f5a976a3f45b62105ff', run_link='', source='models:/m-cdc5477db7844f37aa3caf0cca141c55', status='READY', status_message=None, tags={'estimator name': 'linear regression'}, user_id=None, version=2, workspace='default'>

In [ ]:
import datetime
date = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
client.update_model_version(name=model_name,
                            version=version,
                            description=f"This model version ({version}) was transitioned to {stage} on {date}"
                            )

<ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='Staging', deployment_job_state=None, description='This model version (4) was transitioned to Staging on 2026-04-20 08:26:00', last_updated_timestamp=1776673560653, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:/m-f5ee30b061284ed098a56d94be7870d9', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>

In [ ]:
# Lets say we want to move the current prod model to archived and promote a staged model to prod

### Comparing versions and selecting the new "Production" model

In the last section, we will retrieve models registered in the model registry and compare their performance on an unseen test set. The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:

1. Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2021.
2. Download the `DictVectorizer` that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
3. Preprocess the test set using the `DictVectorizer` so we can properly feed the regressors.
4. Make predictions on the test set using the model versions that are currently in the "Staging" and "Production" stages, and compare their performance.
5. Based on the results, update the "Production" model version accordingly.


**Note: the model registry doesn't actually deploy the model to production when you transition a model to the "Production" stage, it just assign a label to that model version. You should complement the registry with some CI/CD code that does the actual deployment.**

In [51]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(name, stage, X_test, y_test):
    print(f"Loading model {name} from stage {stage}")
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    print(f'Model loaded has run Id {model.metadata.run_id} and model version {model.metadata.get_model_info().tags["mlflow.modelVersions"]}')
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [44]:
test_model = mlflow.pyfunc.load_model(f"models:/{model_name}/Production")
test_model.metadata.get_model_info()

In [17]:
df = read_dataframe("../data/green_tripdata_2021-03.parquet")

In [ ]:
# We would need to download the correct preprocessor for each model we want to asses.
# In this notebook, we just used the same preprocessor for all models, but in real life, we could have different preprocessors for each model.
with open("preprocessor/preprocessor.b", "rb") as f_in:
    dv = pickle.load(f_in)

In [20]:
X_test = preprocess(df, dv)

In [21]:
target = "duration"
y_test = df[target].values

In [52]:
%time test_model(name=model_name, stage="Production", X_test=X_test, y_test=y_test)


Loading model nyc-taxi-regressor from stage Production
Model loaded has run Id 59c9635e7659471cade01e548ad39dfe and model version [{"name": "nyc-taxi-regressor", "version": 1}]
CPU times: user 2.04 s, sys: 10.9 ms, total: 2.05 s
Wall time: 1.5 s


{'rmse': 6.414347681138584}

In [53]:
%time test_model(name=model_name, stage="Staging", X_test=X_test, y_test=y_test)

Loading model nyc-taxi-regressor from stage Staging
Model loaded has run Id c34e8acacfe647a2a82320a6687d6872 and model version [{"name": "nyc-taxi-regressor", "version": 4}]
CPU times: user 55.3 ms, sys: 3.93 ms, total: 59.3 ms
Wall time: 72.7 ms


{'rmse': 12.251808643504843}

In [54]:
# If staged model is better, we could then transition it to Production
client.transition_model_version_stage(
    name=model_name,
    version=4,
    stage="Production",
    archive_existing_versions=True
)

/tmp/ipykernel_4718/2884639092.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1776672773453, current_stage='Production', deployment_job_state=None, description='This model version (4) was transitioned to Staging on 2026-04-20 08:26:00', last_updated_timestamp=1776751646725, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='c34e8acacfe647a2a82320a6687d6872', run_link=None, source='models:/m-f5ee30b061284ed098a56d94be7870d9', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>